In [2]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

df = pd.read_csv('depi_ungrouped_withfeatures.csv')

# Discount %
df['discount_percentage'] = (df['Discount Amount'] / (df['Subtotal'] + df['Discount Amount'])) * 100

# Convert Created at to datetime for sales over time
df['Created at'] = pd.to_datetime(df['Created at'])
df['date'] = df['Created at'].dt.date
df['month'] = df['Created at'].dt.month

# Sales over time
sales_over_time = df.groupby('date')['Total'].sum().reset_index()
sales_time_fig = px.line(sales_over_time, x='date', y='Total', title='Total Sales Over Time')

# Top ordered items
top_items = df.groupby('Lineitem name')['Lineitem quantity'].sum().head(10).sort_values(ascending=False).reset_index()
top_items_fig = px.bar(top_items, x='Lineitem quantity', y='Lineitem name', orientation='h', title='Top 10 Ordered Items by Quantity')

# Most expensive items ordered
top_expensive = df.groupby('Lineitem name').agg({'Lineitem price': 'max', 'Lineitem quantity': 'sum'}).sort_values(by='Lineitem price', ascending=False).head(10).reset_index()
top_expensive_fig = px.bar(top_expensive, x='Lineitem price', y='Lineitem name', color='Lineitem quantity', orientation='h', title='Top 10 Most Expensive Items and Their Quantities')

# Weekend vs Holiday Impact as subplots
weekend_data = df.groupby('is_weekend')['Total'].sum().reset_index()
holiday_data = df.groupby('is_holiday')['Total'].sum().reset_index()
impact_fig = make_subplots(rows=1, cols=2, subplot_titles=('Weekend Impact', 'Holiday Impact'))
impact_fig.add_trace(go.Bar(x=weekend_data['is_weekend'], y=weekend_data['Total'], name='Weekend'), row=1, col=1)
impact_fig.add_trace(go.Bar(x=holiday_data['is_holiday'], y=holiday_data['Total'], name='Holiday'), row=1, col=2)
impact_fig.update_layout(title_text='Sales Impact: Weekend vs Holiday')

# Popular Payment Methods (Top 10)
top_payments = df['Payment Method'].value_counts().head(10).sort_values(ascending=False).reset_index()
top_payments.columns = ['Payment Method', 'Count']
payment_fig = px.bar(top_payments, x='Payment Method', y='Count', title='Top 10 Payment Methods',color='Payment Method')

# Shipping Methods (Top 10)
top_shipping = df['Shipping Method'].value_counts().head(10).sort_values(ascending=False).reset_index()
top_shipping.columns = ['Shipping Method', 'Count']
shipping_fig = px.bar(top_shipping, x='Shipping Method', y='Count', title='Top 10 Shipping Methods',color='Shipping Method')

# Sales by Province (Horizontal)
province_sales = df.groupby('Shipping Province Name')['Total'].sum().sort_values(ascending=False).reset_index()
province_fig = px.bar(province_sales, x='Total', y='Shipping Province Name', orientation='h', title='Total Sales by Province',color='Shipping Province Name')

# Discount Impact
discount_fig = px.scatter(df, x='discount_percentage', y='Lineitem quantity', title='Discount % vs Quantity Sold', hover_data=['Lineitem price'])

# Dropdown for seasonal or monthly analysis
seasonal_month_fig = dcc.Graph(id='season-month-sales')
dropdown = dcc.Dropdown(
    id='season-month-choice',
    options=[
        {'label': 'Season', 'value': 'season'},
        {'label': 'Month', 'value': 'month'}
    ],
    value='season',
    style={'width': '50%','color': 'black'}
)

# App
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("E-commerce Sales Dashboard", style={'textAlign': 'center','color': 'white'}),

    dcc.Graph(figure=sales_time_fig),
    dcc.Graph(figure=top_items_fig),
    dcc.Graph(figure=top_expensive_fig),
    dcc.Graph(figure=impact_fig),
    dcc.Graph(figure=payment_fig),
    dcc.Graph(figure=shipping_fig),
    dcc.Graph(figure=province_fig),
    dcc.Graph(figure=discount_fig),

    html.Div([html.Label("View Most Sales by:"), dropdown, seasonal_month_fig], style={'padding': 20,'color': 'white'})
])

@app.callback(
    Output('season-month-sales', 'figure'),
    Input('season-month-choice', 'value')
)
def update_figure(view):
    if view == 'season':
        data = df.groupby('season')['Total'].sum().sort_values(ascending=False).reset_index()
        fig = px.bar(data, x='season', y='Total', title='Total Sales by Season',color='season')
    else:
        data = df.groupby('month')['Total'].sum().sort_values(ascending=False).reset_index()
        fig = px.bar(data, x='month', y='Total', title='Total Sales by Month',color='month')
    return fig

if __name__ == '__main__':
    app.run(debug=True, port=8054)

In [4]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

df = pd.read_csv('depi_ungrouped_withfeatures.csv')

# Discount %
df['discount_percentage'] = (df['Discount Amount'] / (df['Subtotal'] + df['Discount Amount'])) * 100

# Convert Created at to datetime for sales over time
df['Created at'] = pd.to_datetime(df['Created at'])
df['date'] = df['Created at'].dt.date
df['month'] = df['Created at'].dt.month

# App
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("E-commerce Sales Dashboard", style={'textAlign': 'center'}),

    html.Label("Select Season:"),
    dcc.Dropdown(
        id='season-dropdown',
        options=[{'label': s, 'value': s} for s in sorted(df['season'].dropna().unique())],
        value=sorted(df['season'].dropna().unique())[0],
        style={'width': '50%'}
    ),

    dcc.Graph(id='season-sales-time'),
    dcc.Graph(id='season-top-items'),
    dcc.Graph(id='season-top-expensive'),
    dcc.Graph(id='season-impact'),
])

@app.callback(
    [
        Output('season-sales-time', 'figure'),
        Output('season-top-items', 'figure'),
        Output('season-top-expensive', 'figure'),
        Output('season-impact', 'figure')
    ],
    [Input('season-dropdown', 'value')]
)
def update_dashboard(selected_season):
    season_df = df[df['season'] == selected_season]

    # Sales over time
    sales_over_time = season_df.groupby('date')['Total'].sum().reset_index()
    sales_time_fig = px.line(sales_over_time, x='date', y='Total', title=f'Total Sales Over Time ({selected_season})')

    # Top ordered items
    top_items = season_df.groupby('Lineitem name')['Lineitem quantity'].sum().sort_values(ascending=False).head(10).reset_index()
    top_items_fig = px.bar(top_items, x='Lineitem quantity', y='Lineitem name', orientation='h', title=f'Top 10 Ordered Items ({selected_season})')

    # Most expensive items ordered
    top_expensive = season_df.groupby('Lineitem name').agg({'Lineitem price': 'max', 'Lineitem quantity': 'sum'}).sort_values(by='Lineitem price', ascending=False).head(10).reset_index()
    top_expensive_fig = px.bar(top_expensive, x='Lineitem price', y='Lineitem name', color='Lineitem quantity', orientation='h', title=f'Top 10 Most Expensive Items ({selected_season})')

    # Weekend vs Holiday Impact
    weekend_data = season_df.groupby('is_weekend')['Total'].sum().reset_index()
    holiday_data = season_df.groupby('is_holiday')['Total'].sum().reset_index()
    impact_fig = make_subplots(rows=1, cols=2, subplot_titles=('Weekend Impact', 'Holiday Impact'))
    impact_fig.add_trace(go.Bar(x=weekend_data['is_weekend'], y=weekend_data['Total'], name='Weekend'), row=1, col=1)
    impact_fig.add_trace(go.Bar(x=holiday_data['is_holiday'], y=holiday_data['Total'], name='Holiday'), row=1, col=2)
    impact_fig.update_layout(title_text=f'Sales Impact: Weekend vs Holiday ({selected_season})')

    return sales_time_fig, top_items_fig, top_expensive_fig, impact_fig

if __name__ == '__main__':
    app.run(debug=True)
